# PathSnap · One trip from GPS to road segments

Each observation needs candidate roads for inference and an independently derived label for evaluation. Start with trip 00000001. Run the cells in order from the project folder. Set PATHSNAP_DATA to use another dataset location.

In [1]:
from pathlib import Path
import numpy as np
import folium
from roads import ROOT, load_trip, candidates, reference_labels, distances

trip = ROOT / "train" / "00000001"
nodes, arcs, track = load_trip(trip)
route = np.loadtxt(trip / f"{trip.name}.route", dtype=int, ndmin=1)
assert np.isfinite(track).all() and np.isfinite(nodes).all()
assert arcs.min() >= 0 and arcs.max() < len(nodes)
assert route.min() >= 0 and route.max() < len(arcs)
dt = np.diff(track[:, 2])
print(f"{len(track):,} GPS readings · {len(nodes):,} nodes · {len(arcs):,} arcs · {len(route):,} route entries")
print(f"Time gaps over 1 second: {(dt > 1).sum()}. Longest gap: {dt.max():.0f} seconds.")

1,315 GPS readings · 88,809 nodes · 165,790 arcs · 525 route entries
Time gaps over 1 second: 25. Longest gap: 16 seconds.


## Generate candidates and evaluate the nearest road

The spatial index only sees GPS readings and the road network. The reference alignment separately minimizes total point-to-segment distance along nondecreasing positions in the supplied route. It allows repeated assignments and skipped segments. At shared endpoints and during stops several alignments may be equally plausible. Exact-ID accuracy therefore measures agreement with this labeling rule.

In [2]:
ids, features, neighbor_features, geometry = candidates(nodes, arcs, track)
labels = reference_labels(route, geometry)
correct = ids[:, 0] == labels
covered = (ids == labels[:, None]).any(axis=1)
print(f"Nearest road: {correct.sum()}/{len(track)} correct ({correct.mean():.2%})")
print(f"Five-candidate miss rate: {(~covered).sum()}/{len(track)} ({1-covered.mean():.2%})")
gap = np.expm1(features[:,1,0]) - np.expm1(features[:,0,0])
print(f"Two closest roads within 1 metre of each other in distance: {(gap < 1).sum()} readings")
obs, starts, vec = geometry
for i in np.linspace(0,len(track)-1,16,dtype=int):
    d = distances(obs[i],starts,vec)
    expected = np.lexsort((np.arange(len(d)), d))[:5]
    assert np.array_equal(ids[i], expected)
print("Verified the five candidate IDs by exhaustive search at 16 readings.")

Nearest road: 1053/1315 correct (80.08%)
Five-candidate miss rate: 10/1315 (0.76%)
Two closest roads within 1 metre of each other in distance: 332 readings
Verified the five candidate IDs by exhaustive search at 16 readings.


## Inspect the map

Toggle layers in the top-right corner. Purple markers identify ambiguous nearest-road errors. Click one to compare its nearest arc, inferred reference arc, and distance margin. The gray layer shows roads returned by candidate search along this trip.

In [3]:
m = folium.Map(prefer_canvas=True)
def segment_layer(name, arc_ids, color, weight, show=True):
    layer = folium.FeatureGroup(name=name, show=show).add_to(m)
    lines = nodes[arcs[np.unique(arc_ids)]][:,:,::-1].tolist()
    folium.PolyLine(lines,color=color,weight=weight,opacity=.72).add_to(layer)

segment_layer("Candidate road network", ids, "#64748b", 2)
segment_layer("Reference route", route, "#15803d", 6)
segment_layer("Nearest-road prediction", ids[:,0], "#ea580c", 3)
gps = folium.FeatureGroup(name="GPS observations").add_to(m)
for lon,lat,t in track:
    folium.CircleMarker([lat,lon],radius=2,color="#0369a1",weight=1,fill=True,tooltip=f"{t:g} s").add_to(gps)
review = folium.FeatureGroup(name="Ambiguous nearest-road errors").add_to(m)
ambiguous = np.flatnonzero((gap < 1) & ~correct)
for i in ambiguous[::max(1,len(ambiguous)//20)]:
    lon,lat,t = track[i]
    folium.CircleMarker([lat,lon],radius=5,color="#7e22ce",popup=f"Reading {i}, {t:g} s. Nearest arc {ids[i,0]}, reference {labels[i]}. Distance margin {gap[i]:.2f} m.").add_to(review)
m.fit_bounds([track[:,:2].min(0)[::-1].tolist(),track[:,:2].max(0)[::-1].tolist()])
folium.LayerControl(collapsed=False).add_to(m)
out = Path("results") / "trip_00000001.html"
out.parent.mkdir(exist_ok=True)
m.save(out)
print(f"Interactive map saved to {out}.")

Interactive map saved to results/trip_00000001.html.
